In [ ]:
import requests
import os
import pandas as pd
from connec_functions import GDB

from accessibility import check_endpoint

### Interoperability Assessment of EuroArgo SPARQL Endpoint

### Endpoint

In [ ]:
# The given endpoint is the human-facing (UI) SPARQL Endpoint
browser_endpoint = "https://co.ifremer.fr/co/argo-linked-data/html/Argo-HTML-SPARQL/"

In [ ]:
# The machine-facing (API) SPARQL Endpoint is:
cli_endpoint = "https://sparql.ifremer.fr/argo/query"
gdb = GDB(cli_endpoint, "endpoint_queries")

##### Table Of Content

- [Understanding the SPARQL endpoint structure](#understanding-sparql-endopint-structure)
- [Technical interoperability](#technical-interoperability)
    - SPARQL protocol support ~ version and supported features
    - Content negotiation and format support 
    - Endpoint connectivity and performance (~ includes endpoint availability, connectivity, perofrmance and timeout handling) 
- [Semantic interoperability](#semantic-interoperability)
    - Use of standard vocabularies
    - Ontology usage and semantic alignment 
    - Use of linked data principles (URIs)
    - Multilingual support
    - (Inference and reasoning support)



### Understanding the SPARQL endpoint structure
*Involves getting to understand the underlying RDF graph structure*  

In [ ]:
# Test browser/UI endpoint vs. CLI/API endpoint
params = {
    "query": "SELECT * WHERE { ?s ?p ?o } LIMIT 1"
}
headers = {
    "Accept": "application/sparql-results+json"
}

resp_ui = requests.get(browser_endpoint, params=params, headers=headers)
print("UI endpoint status:", resp_ui.status_code)

resp_api = requests.get(cli_endpoint, params=params, headers=headers)
print("API endpoint status:", resp_api.status_code)

In [ ]:
# general exploration
gdb.execute_to_df("general.sparql")

,s,p,o
0,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/ns/dcat#Catalog
1,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://co.ifremer.fr/co/argo-linked-data/doc/...
2,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/2000/01/rdf-schema#label,aoml
3,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/description,\n Catalog of the Argo data...
4,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/publisher,http://www.argodatamgt.org/Data-Mgt-Team/ADMT-...
5,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/title,aoml Argo DAC metadata
6,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
7,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
8,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
9,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...


The underlying RDF graph structure was examined, through both the cli_endpoint and browser_endpoint, by sequentially executing multiple SPARQL queries and documented in the following diagram:

[euroargo datamodel diagram](./images/euroargo_datamodel-Dataset_Platform_ArgoFloat.drawio.png)

following diagram of datamodel available through browser_endpoint:

[argo_ontology_prototype.png](https://co.ifremer.fr/co//argo-linked-data/doc/argo_ontology_prototype.png)

##### Note on 'granularity' ....

- data granularity: file level 
  - data files are 'accessible' with dcat:downloadUrl within a dcat:distribution
  (datapoints not described in RDF - to use data itself the file needs to be downloaded and data needs to be read into memory)

### Technical interoperability

#### SPARQL protocol support 
*Involves checking the version and supported features*

To fully comply with the SPARQL 1.1 Protocol (per [W3C spec](https://www.w3.org/TR/sparql11-protocol/)), an endpoint must support specific HTTP request types (methods) and media types for two main functionalities:

1. SPARQL Query (Read operations)

(Support for SELECT, ASK, DESCRIBE, CONSTRUCT)
| HTTP Method | Content-Type                        | Required                             |
| ----------- | ----------------------------------- | ------------------------------------ |
| **GET**     | N/A (query via URL)                 | ✔️ Yes                               |
| **POST**    | `application/x-www-form-urlencoded` | ✔️ Yes                               |
| **POST**    | `application/sparql-query`          | ⚠️ Optional, but strongly encouraged |


- Accept headers for content negotiation should also be supported:
    - application/sparql-results+json
    - application/sparql-results+xml
    - application/rdf+xml, text/turtle, etc. for CONSTRUCT/DESCRIBE


2. SPARQL Update (Write operations)

(Support for INSERT, DELETE, LOAD, etc.)
| HTTP Method | Content-Type                | Required |
| ----------- | --------------------------- | -------- |
| **POST**    | `application/sparql-update` | ✔️ Yes   |


- Only POST is allowed for SPARQL Update — GET is not valid.
- Responses are typically empty or with status 204 No Content.


3. Content Negotiation

The endpoint must respond appropriately based on the Accept header:
- For SELECT, ASK: support application/sparql-results+json, ...+xml
- For CONSTRUCT, DESCRIBE: support at least one RDF serialization:
    - application/rdf+xml, text/turtle, application/ld+json, etc.


4. Response Codes and Compliance

- Return 200 OK for valid queries.
- Return 400 Bad Request or 500 Internal Server Error for syntax/logic errors.
- May support OPTIONS for preflight (CORS-related, not required by SPARQL spec).



Summary Table:
| Function            | HTTP Method | Content-Type                        | Required                        |
| ------------------- | ----------- | ----------------------------------- | ------------------------------- |
| Query (GET)         | `GET`       | URL parameter `?query=`             | ✔️                              |
| Query (POST)        | `POST`      | `application/x-www-form-urlencoded` | ✔️                              |
| Query (POST raw)    | `POST`      | `application/sparql-query`          | ⚠️ Optional                     |
| Update              | `POST`      | `application/sparql-update`         | ✔️                              |
| Content Negotiation | `GET/POST`  | `Accept` header for result formats  | ✔️                              |
| Preflight/CORS      | `OPTIONS`   | N/A                                 | ❌ Not part of spec, but helpful |



In [ ]:
def test_read_operations(endpoint, read_type="SELECT"):
    print(f"📥 Testing SPARQL 1.1 READ Operation: {read_type}\n")

    queries = {
        "SELECT": "SELECT * WHERE { ?s ?p ?o } LIMIT 1",
        "ASK": "ASK { ?s ?p ?o }",
        "CONSTRUCT": "CONSTRUCT { <http://example.org/res> ?p ?o } WHERE { <http://example.org/res> ?p ?o }",
        "DESCRIBE": "DESCRIBE <http://example.org/res>"
    }

    if read_type not in queries:
        print(f"❌ Unsupported query type: '{read_type}'. Choose from: {list(queries.keys())}")
        return

    query = queries[read_type]

    # Adjust Accept header based on type of expected result
    accept = (
        "application/sparql-results+json"
        if read_type in ["SELECT", "ASK"]
        else "text/turtle"
    )

    try:
        r_get = requests.get(endpoint, params={"query": query}, headers={"Accept": accept})
        print("GET:", r_get.status_code, r_get.headers.get("Content-Type"))
    except Exception as e:
        print("GET Error:", e)

    try:
        r_post_form = requests.post(endpoint, data={"query": query}, headers={"Accept": accept})
        print("POST form:", r_post_form.status_code, r_post_form.headers.get("Content-Type"))
    except Exception as e:
        print("POST form Error:", e)

    try:
        r_post_raw = requests.post(endpoint, data=query, headers={
            "Content-Type": "application/sparql-query",
            "Accept": accept
        })
        print("POST raw:", r_post_raw.status_code, r_post_raw.headers.get("Content-Type"))
    except Exception as e:
        print("POST raw Error:", e)

In [ ]:
test_read_operations(cli_endpoint, "SELECT")

In [ ]:
test_read_operations(cli_endpoint, "ASK")

In [ ]:
test_read_operations(cli_endpoint, "CONSTRUCT")

In [ ]:
test_read_operations(cli_endpoint, "DESCRIBE")

In [ ]:
def test_write_operations(endpoint_url, operation_type="INSERT DATA"):
    updates = {
        "INSERT DATA": """
            INSERT DATA {
                <http://example.org/test> <http://example.org/prop> "test_insert" .
            }
        """,
        "DELETE DATA": """
            DELETE DATA {
                <http://example.org/test> <http://example.org/prop> "test_insert" .
            }
        """,
        "DELETE/INSERT": """
            DELETE {
                <http://example.org/test> <http://example.org/prop> "old_value" .
            }
            INSERT {
                <http://example.org/test> <http://example.org/prop> "new_value" .
            }
            WHERE {
                OPTIONAL { <http://example.org/test> <http://example.org/prop> "old_value" }
            }
        """,
        "CLEAR DEFAULT": """
            CLEAR DEFAULT
        """
    }

    headers = {
        "Content-Type": "application/sparql-update"
    }

    print(f"📝 Testing SPARQL 1.1 Write Operation: {operation_type}\n")

    if operation_type not in updates:
        print(f"❌ Unsupported operation type: '{operation_type}'. Choose from: {list(updates.keys())}")
        return

    update_query = updates[operation_type]

    try:
        response = requests.post(endpoint_url, data=update_query.strip(), headers=headers, timeout=10)
        print(f"Status: {response.status_code} | Content-Type: {response.headers.get('Content-Type')}")
        if response.status_code >= 400:
            print("⚠️  Error response:", response.text[:200])
    except Exception as e:
        print("❌ Request failed:", e)



In [ ]:
test_write_operations(cli_endpoint, "INSERT DATA")

In [ ]:
test_write_operations(cli_endpoint, "DELETE DATA")

In [ ]:
test_write_operations(cli_endpoint, "DELETE/INSERT")

In [ ]:
test_write_operations(cli_endpoint, "CLEAR DEFAULT")

#### Content negotiation and format support 

both human-faced and machine-faced SPARQL endpoint were consulted for the assessment of this part


In [13]:
def get_header_info(endpoint):
    response = requests.get(endpoint)
    server_headers = response.headers

    return {
        # General protocol and status checks
        "General protocol and status checks" : 
            {
            "HTTP/HTTPS Protocol": endpoint.startswith("http"),
            "Status Code 200 OK": response.status_code == 200,
            },
        # Content negotiation
        "Content negotiation" : 
            {
            "MIME Type Present": "Content-Type" in server_headers,
            "MIME Type": server_headers.get("Content-Type"),
            "Content Negotiation Support": "Vary" in server_headers or "Accept" in server_headers,
            "Vary": server_headers.get("Vary"),                         # response may change depending on 'Accept-Encoding'
            "Accept-Ranges": server_headers.get("Accept-Ranges"),       # support partial content requests (e.g., 'bytes')
            },
        # Content negotiation related headers
        "Content negotiation related headers" : 
            { 
            "Content-Length": server_headers.get("Content-Length"),             # total size of the file, if specified
            "Content-Disposition": server_headers.get("Content-Disposition"),   # tells if file is inline or attachment
            "Content-Encoding": server_headers.get("Content-Encoding"),         # compression format (e.g., gzip)
            "Accept": server_headers.get("Accept"),                             # what media types the client accepts (rare in response)
            "Accept-Encoding": server_headers.get("Accept-Encoding"),           # what compression formats the client accepts (rare in response)
            },
        # Caching-related headers
        "Caching-related headers" : 
            {
            "Cache-Control": server_headers.get("Cache-Control"),       # caching policy (e.g., no-cache, max-age)
            "ETag": server_headers.get("ETag"),                         # version identifier for the file, useful for caching and validation
            }
    }

In [14]:
get_header_info(browser_endpoint)

{'General protocol and status checks': {'HTTP/HTTPS Protocol': True,
  'Status Code 200 OK': True},
 'Content negotiation': {'MIME Type Present': True,
  'MIME Type': 'text/html',
  'Content Negotiation Support': True,
  'Vary': 'Accept-Encoding',
  'Accept-Ranges': 'bytes'},
 'Content negotiation related headers': {'Content-Length': '2756',
  'Content-Disposition': None,
  'Content-Encoding': 'gzip',
  'Accept': None,
  'Accept-Encoding': None},
 'Caching-related headers': {'Cache-Control': None,
  'ETag': '"42124b50-2284-5c4e0d31060f4"'}}

In [15]:
get_header_info(cli_endpoint)

{'General protocol and status checks': {'HTTP/HTTPS Protocol': True,
  'Status Code 200 OK': False},
 'Content negotiation': {'MIME Type Present': True,
  'MIME Type': 'text/plain;charset=utf-8',
  'Content Negotiation Support': True,
  'Vary': 'Accept,Accept-Encoding,Accept-Charset,Origin,Access-Control-Request-Method,Access-Control-Request-Headers',
  'Accept-Ranges': None},
 'Content negotiation related headers': {'Content-Length': '33',
  'Content-Disposition': None,
  'Content-Encoding': None,
  'Accept': None,
  'Accept-Encoding': None},
 'Caching-related headers': {'Cache-Control': 'must-revalidate,no-cache,no-store',
  'ETag': None}}

In [19]:
# assess formats returned by cli_endpoint
query = "SELECT * WHERE { ?s ?p ?o } LIMIT 10"
formats = {
    "SPARQL JSON": "application/sparql-results+json",
    "SPARQL XML": "application/sparql-results+xml",
    "RDF/XML": "application/rdf+xml",
    "Turtle": "text/turtle",
    "JSON-LD": "application/ld+json"
}

for name, mime in formats.items():
    headers = {"Accept": mime}
    params = {"query": query}
    
    try:
        r = requests.get(cli_endpoint, headers=headers, params=params, timeout=10)
        status = r.status_code
        ctype = r.headers.get("Content-Type", "")
        print(f"{name:15} → {status} | {ctype}")
    except Exception as e:
        print(f"{name:15} → ERROR: {e}")

SPARQL JSON     → 200 | application/sparql-results+json; charset=utf-8
SPARQL XML      → 200 | application/sparql-results+xml
RDF/XML         → 200 | application/sparql-results+xml
Turtle          → 200 | application/sparql-results+xml
JSON-LD         → 200 | application/sparql-results+xml


formats returned by `browser_endpoint` include:
- text → correct
- JSON → correct
- XML → need to check 
- CSV → correct
- TSV → correct

#### Endpoint connectivity and performance 
*Includes checking endpoint availability, connectivity, performance and timeout handling*


(aspect of technical interoperability, no heavy focus on this)

In [24]:
if check_endpoint(cli_endpoint):
    print("The endpoint is machine-accessible.")
else:
    print("The endpoint is not machine-accessible.")

Checking endpoint: https://sparql.ifremer.fr/argo/query
Failed to access endpoint: 404
The endpoint is not machine-accessible.


In [25]:
import requests
import time

# The endpoint you want to test
url = "https://sparql.ifremer.fr/argo/query"

# Optional: a simple test payload (SPARQL query in POST body, for example)
# This depends on the endpoint API; here we just send an empty or small query
payload = {
    "query": "SELECT * WHERE { ?s ?p ?o } LIMIT 10"  # simple SPARQL query
}
headers = {
    "Accept": "application/sparql-results+json"
}

# Configuration
timeout_seconds = 60

def test_endpoint(url, payload=None, headers=None, timeout=5):
    results = {}
    
    start_time = time.time()
    try:
        response = requests.post(url, data=payload, headers=headers, timeout=timeout)
        elapsed = time.time() - start_time

        results["reachable"] = True
        results["status_code"] = response.status_code
        results["response_time_seconds"] = round(elapsed, 3)
        results["success"] = response.ok
        results["content_preview"] = response.text[:200]  # first 200 chars
    except requests.exceptions.Timeout:
        results["reachable"] = False
        results["error"] = f"Request timed out after {timeout} seconds"
    except requests.exceptions.ConnectionError as e:
        results["reachable"] = False
        results["error"] = f"Connection error: {e}"
    except Exception as e:
        results["reachable"] = False
        results["error"] = str(e)

    return results

# Run test
result = test_endpoint(url, payload=payload, headers=headers, timeout=timeout_seconds)
result

{'reachable': True,
 'status_code': 200,
 'response_time_seconds': 0.128,
 'success': True,
 'content_preview': '{ "head": {\n    "vars": [ "s" , "p" , "o" ]\n  } ,\n  "results": {\n    "bindings": [\n      { \n        "s": { "type": "uri" , "value": "https://argo.ucsd.edu/data/data-from-gdacs#aoml" } ,\n        "p": {'}

Findings

SPARQL protocol support
- support for GET(?) and POST method (POST only retrieves this is a common observation in public-facing SPARQL endpoints)

Content negotiation and format support
- human-facing SPARQL endpoint supports various file formats: Text, JSON, XML, CSV, TSV  
- machine-facing SPARQL endpoint support main RDF formats: JSONLD, Turtle and RDF/XML 

Endpoint connectivity and performance
- cli_endpoint is available, 
- cli_endpoint can't handle many requests (had to introduce function parameters and run this notebook over large period, otherwise would get timeout-errors) 
- browser_endpoint has issues with large SPARQL queries (e.g. when LIMIT not specified: SELECT ?subject ?predicate ?object WHERE { ?subject ?predicate ?object })


to include/integrate with findings on technical interoperability above:  
- simple / general / large SPARQL queries result in a 502 error

### Semantic interoperability

other provided links that are relevant in this context:
- [Argo vocabulary server](https://vocab.nerc.ac.uk/search_nvs/)  
- Argo ontology: [https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl](https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl)  
https://service.tib.eu/webvowl/#file=argo-floats.ttl#

#### Use of standard vocabularies

- Commonly accepted vocabularies/ontolgoies are being used 
    - `prefix geo: <https://www.w3.org/2003/01/geo/wgs84_pos#>`  
      `prefix owl: <http://www.w3.org/2002/07/owl#>`  
      `prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>`   
      `prefix ssn: <http://www.w3.org/ns/ssn/>`   
      `prefix xml: <http://www.w3.org/XML/1998/namespace>`   
      `prefix xsd: <http://www.w3.org/2001/XMLSchema#>`    
      `prefix argo: <http://www.argodatamgt.org/argo-ontology#>`   
      `prefix foaf: <http://xmlns.com/foaf/0.1/>`   
      `prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>`   
      `prefix sosa: <http://www.w3.org/ns/sosa/>`   
      `prefix nerc: <http://vocab.nerc.ac.uk/collection/>`   
      `prefix dct: <http://purl.org/dc/terms/>`   
      `prefix prov: <https://www.w3.org/TR/prov-o/>`  

- There are also many references to skos:Concepts in `http://vocab.nerc.ac.uk/collection/...` namespace 


note on standard terms (inside-institute vs. outside-institute)  
...  


#### Ontology usage and semantic alignment 
*Includes 'ontology and vocabulary compatibility', 'cross-dataset semantic linking', 'consistency of semantics'*

- Ontology and vocabulary compatibility  
https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl  
--> description of owl:ontology with identifier <http://www.argodatamgt.org/argo-ontology#>  
--> prefix used in sparql endpoint: prefix argo: <http://www.argodatamgt.org/argo-ontology#>   
--> urls of some predicates in query results:  
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#type> 
   -  <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#datamode> 
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#age>
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#stationData> 
   - ...  

--> urls of some classes in query results:  
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#ActivityData>

==> all these urls redirect to the description of the owl:Ontology  
ActivityData class has URI  http://www.argodatamgt.org/argo-ontology#ActivityData in the owl:Ontology

- Cross-dataset semantic linking  
    - no resources that use skos:exactMatch, etc. to algin with external datasets  
    - ?s ?p ?o --> ?o often skos:Concept from an external collection   

- consistency of semantics  
most classes are described with correct predicates (e.g. dcat:Distribution has properties dcat:downloadURL, etc.)
    - 'group' entities (e.g.  <https://fleetmonitoring.euro-argo.eu/float/1900045#group1>) are not of some class -rather a blank node with the same predicates each time 
    - entities of class dct:Software (e.g.  <https://argo.ucsd.edu/data/argo-software-tools#WJO>) don't have any properties  
    rather:  
    https://argo.ucsd.edu/data/argo-software-tools  
    --> website that describes the argo visualization and access tools & Quality control tools  
    --> information not available in RDF  
    --> note software entity <https://argo.ucsd.edu/data/argo-software-tools/#WJO> is not found/described on that website...  
    similarly:  
    dct:publisher of a dcat:Dataset is 
    http://www.argodatamgt.org/ADMT/ADMT-and-Executive-Board
    --> website that describes the ADMT and Executive Board
    --> information not available in RDF

- some terms (classes and predicates) could be described by already existing terms in common/general/high level ontologies  

- few occurences of typo: dcat:distribution when this represents a RDF class  

- the ontology should be "generalized" ~ aligned with other domain stakeholders and be made available under a more RI neutral namespace ~ to facilitate uptake of terms by different RI's  
--> consortium/workgroup that discussed and determines these tasks and acts as an independent autorative publisher.  


#### Use of linked data principles (URIs)

- very good

In [21]:
"""
SELECT DISTINCT (ISIRI(?s) AS ?isIRI)
WHERE {
  ?s ?p ?o .
}
LIMIT 100
"""
# UI SPARQL endpoint (browser_endpoint) not performant enough to execute this query

'\nSELECT DISTINCT (ISIRI(?s) AS ?isIRI)\nWHERE {\n  ?s ?p ?o .\n}\nLIMIT 100\n'

#### Multilingual support

- no use of language tags spotted (e.g. titles of datasets, ...)
- no support for different language 

In [ ]:
"""
SELECT DISTINCT (LANG(?label) AS ?lang)
WHERE {
  ?s rdfs:label ?label .
  FILTER(LANG(?label) != "")
}
LIMIT 100
"""
# query returns empty results with UI SPARQL endpoint (browser_endpoint)

to include / integrate with findings on semantics interoperability above:  

- there is linking to externally defined standard terms = good
    - leveraging use of linked data 
- usage of internally defined predicates = good, but less good (see point 2)
    - not known by external machines (e.g. dct:title vs ifremer:thisisourname)
    - solution would require community effort to develop standard data model for described entity kinds
- identifier for publisher information can be improved, for example 'http://www.argodatamgt.org/Data-Mgt-Team/ADMT-team-and-Executive-Committee'
  - could alternatively use ROR-ID for institutes, ORC-ID for people
  - currently links to just html page (no ttl or json-ld with content negotiation), could be described as linked data
- https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#Datacenter not in the described ontology

